# 00 - Deep Mathematical Baseline & Fair Checking

This foundational notebook implements the exact 40/60 momentum formula and dynamic affection-rate banding to create perfectly contextual targets for classification. It ensures that all 12 subsequent machine learning models are evaluated on an identical, strictly verified foundation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the purely chronological splits from Phase 1
train_df = pd.read_csv('../../../data/processed/train.csv')
val_df = pd.read_csv('../../../data/processed/val.csv')

print(f"Train Shape: {train_df.shape} | Val Shape: {val_df.shape}")

Train Shape: (350, 35) | Val Shape: (75, 35)


### Step 1 & 2: Base Momentum and Affection Rates ($W$)
We calculate the 40/60 base momentum, and then rigorously scan the training dataset to calculate the mean positive/negative impact (Affection Rate) of every single binary categorical variable.

In [2]:
global_mean = train_df['Attendance_Percentage'].mean()
print(f"Global Average Attendance (Train): {global_mean:.2f}%\n")

# 1. Base Momentum Formula (40% Long-term, 60% Short-term)
def calculate_momentum(df):
    return (0.40 * df['Monthly_Expanding_Mean']) + (0.60 * df['Rolling_Avg_3_Lectures'])

train_df['Base_Momentum'] = calculate_momentum(train_df)

# 2. Identify all binary context fields (Weather, Time, Subject, Exams, etc.)
binary_cols = [
    'Is_Post_Lunch_Class', 'Is_Holiday_Adjacent', 'Week_Before_Exam_Flag'
] + [c for c in train_df.columns if c.startswith(('Weather_', 'Subject_', 'Day_of_Week_', 'Practical_Theory_', 'Time_of_Day_Cluster_'))]

# 3. Calculate Affection Rates (W) exclusively from Training Data
affection_rates = {}
print("--- Extracted Affection Rates (Weights) ---")
for col in binary_cols:
    if col in train_df.columns:
        # The mean attendance when this factor is present
        mean_when_present = train_df[train_df[col] == 1]['Attendance_Percentage'].mean()
        # The weight is the difference from the global average
        weight = mean_when_present - global_mean
        # Handle NaN if a category never appeared (rare)
        affection_rates[col] = weight if not pd.isna(weight) else 0.0
        
        if affection_rates[col] != 0:
            print(f"{col.ljust(35)}: {affection_rates[col]:>6.2f}%")

Global Average Attendance (Train): 19.81%

--- Extracted Affection Rates (Weights) ---
Is_Post_Lunch_Class                :  -0.38%
Day_of_Week_Monday                 :  -1.48%
Day_of_Week_Saturday               :  -4.39%
Day_of_Week_Thursday               :   3.67%
Day_of_Week_Tuesday                :  -0.46%
Day_of_Week_Wednesday              :   0.86%
Subject_Data Science & Machine Learning:   1.11%
Subject_Industry Readiness Program :   0.33%
Subject_Innovation and Entrepreneurship Development:  -1.23%
Subject_MAD Practical              :  -3.86%
Subject_Mini Project               :  -5.44%
Subject_Mobile Application Development:   1.67%
Subject_Principles of Cloud Management and Security:   0.79%
Subject_STQA Practical             :  -2.42%
Subject_Software Testing and Quality Assurance:  -0.68%
Weather_Sunny                      :  -0.25%
Practical_Theory_Theory            :   0.92%
Time_of_Day_Cluster_Morning        :   0.08%


### Step 3: Expected Attendance ($E_i$) & Residual Calculation ($R_i$)
We sum the Base Momentum and all applicable Affection Rates for every single lecture to form the `Expected_Attendance`.
Then, we calculate the Residual (Actual - Expected).

In [3]:
def calculate_expected_and_residual(df, rates):
    # Start with Base Momentum
    expected = calculate_momentum(df)
    
    # Add the affection rate for every factor that is present (value == 1)
    for col in rates.keys():
        if col in df.columns:
            expected += (df[col] * rates[col])
            
    # Mathematically clip constraints
    expected = expected.clip(0, 100)
    
    # Calculate Residual
    residual = df['Attendance_Percentage'] - expected
    return expected, residual

train_df['Expected_Attendance'], train_df['Residual'] = calculate_expected_and_residual(train_df, affection_rates)

### Step 4: Dynamic Tertile Banding
We use Quantiles on the Training Residuals to set the exact boundaries for High, Medium, and Low. This completely eliminates class imbalance.

In [4]:
# qcut divides the residuals into 3 equal buckets
train_df['Attendance_Class'], residual_bins = pd.qcut(train_df['Residual'], q=3, labels=['Low', 'Medium', 'High'], retbins=True)

print("--- Residual Boundaries (Derived from Train) ---")
print(f"Low Band (Underperformed):    Residual <  {residual_bins[1]:.2f}%")
print(f"Medium Band (As Expected):    Residual >= {residual_bins[1]:.2f}% AND <= {residual_bins[2]:.2f}%")
print(f"High Band (Overperformed):    Residual >  {residual_bins[2]:.2f}%")

print("\n--- Class Distribution (Train) ---")
print(train_df['Attendance_Class'].value_counts(normalize=True))

--- Residual Boundaries (Derived from Train) ---
Low Band (Underperformed):    Residual <  -4.10%
Medium Band (As Expected):    Residual >= -4.10% AND <= 2.78%
High Band (Overperformed):    Residual >  2.78%

--- Class Distribution (Train) ---
Attendance_Class
Low       0.335443
Medium    0.332278
High      0.332278
Name: proportion, dtype: float64


### Step 5: Leakage Prevention (Applying to Validation Set)
We apply the exact same formula, exact same Affection Rates, and exact same Cutoffs to the Validation Set. The validation set is NEVER allowed to influence the weights.

In [5]:
val_df['Expected_Attendance'], val_df['Residual'] = calculate_expected_and_residual(val_df, affection_rates)

# Use pd.cut with the exact bins learned from training
residual_bins[0] = -np.inf
residual_bins[-1] = np.inf
val_df['Attendance_Class'] = pd.cut(val_df['Residual'], bins=residual_bins, labels=['Low', 'Medium', 'High'], include_lowest=True)

print("--- Class Distribution (Validation) ---")
print(val_df['Attendance_Class'].value_counts(normalize=True))

--- Class Distribution (Validation) ---
Attendance_Class
Low       0.420290
High      0.318841
Medium    0.260870
Name: proportion, dtype: float64


### Step 6: Target Separation & Standard Scaling
We separate the targets (`y`) from the features (`X`), and apply a `StandardScaler` to ensure algorithms like SVM and KNN are judged fairly.

In [6]:
targets_to_drop = [
    'Attendance_Percentage', 'Attendance_Class', 'Expected_Attendance', 'Residual', 'Base_Momentum',
    'Students_Present', 'Total_Enrolled', 'Date', 'Start_Time', 'End_Time',
    'Faculty_ID', 'Semester', 'Branch', 'Section', 'Classroom', 'Special_Event'
]

train_df = train_df.dropna(subset=['Attendance_Class']).reset_index(drop=True)
val_df = val_df.dropna(subset=['Attendance_Class']).reset_index(drop=True)
y_train_reg = train_df['Attendance_Percentage']
y_train_class = train_df['Attendance_Class']
X_train = train_df.drop(columns=[col for col in targets_to_drop if col in train_df.columns]).select_dtypes(exclude=['object', 'string'])

y_val_reg = val_df['Attendance_Percentage']
y_val_class = val_df['Attendance_Class']
X_val = val_df.drop(columns=[col for col in targets_to_drop if col in val_df.columns]).select_dtypes(exclude=['object', 'string'])

# Standard Scaling (Fit strictly on Train!)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)

# Export locked datasets to data/processed for Models 01-12 to consume
X_train_scaled.to_csv('../../../data/processed/X_train_scaled.csv', index=False)
X_val_scaled.to_csv('../../../data/processed/X_val_scaled.csv', index=False)
y_train_reg.to_csv('../../../data/processed/y_train_reg.csv', index=False)
y_val_reg.to_csv('../../../data/processed/y_val_reg.csv', index=False)
y_train_class.to_csv('../../../data/processed/y_train_class.csv', index=False)
y_val_class.to_csv('../../../data/processed/y_val_class.csv', index=False)

print("\n✅ Baseline mathematically established. Scaled datasets exported for Models 01 through 12.")


✅ Baseline mathematically established. Scaled datasets exported for Models 01 through 12.
